# Dynamic Agents in LangChain (Summary)

## 📌 Core Idea
In real-world applications, users are not the same:
- Different languages (English, French, etc.)
- Different roles (internal vs external)
- Different access levels and needs

➡️ A static "one-size-fits-all" agent is not enough.

**Solution:** Build a **Dynamic Agent** that can adapt during runtime.

---

## 🔥 What is a Dynamic Agent?
An agent that can dynamically change:
- Prompt (instructions)
- Tools (what it can use)
- Model (which LLM it runs on)

➡️ These changes can happen **during the conversation**, even multiple times.

---

## 🧠 How It Works: Middleware

### 1. Node-Style Middleware
- Works with: `state` and `runtime`
- Used for:
  - Modifying messages
  - Trimming conversation history

❗ Limitation: Cannot modify the model itself

---

### 2. Wrap-Style Middleware (Key Concept 🔥)
- Works with: `ModelRequest`
- Gives full control over:
  - System prompt
  - Tools
  - Model
  - State

➡️ You are directly modifying how the model behaves

---

## 📦 ModelRequest
Contains:
- `system_prompt`
- `tools`
- `model`
- `state`

➡️ Modifying this = changing the agent’s behavior

---

## 🧪 Key Examples

### 1. Dynamic Language Switching
- Detect user's preferred language
- Switch prompt accordingly

➡️ Example:
- English user → respond in English
- French user → respond in French

---

### 2. Dynamic Tools Based on User Role
- Internal user:
  - Access to database + web search
- External user:
  - Web search only

➡️ Middleware modifies:
```python
model_request.tools
```
---
### 3. Dynamic Model Switching

- Short conversation → small / cheap model  
- Long conversation → larger / more powerful model  

### Example Logic

```python
if message_count > 10:
    use_large_model()
else:
    use_small_model()

In [1]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "dynamic models"

In [2]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.3 MB/s eta 0:00:00


In [3]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# dynamic prompts

In [4]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [5]:
@dataclass
class LanguageContext:
    user_language: str = "English"


In [6]:
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."

    if user_language == "English":
        return base_prompt

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia duit! Tá mé go maith, go raibh maith agat. Agus tú féin?


In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

Hola, estoy bien, ¿y tú?


In [10]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour, je vais bien, merci. Et vous ?
